In [1]:
import sys
import os

module_path = os.path.abspath(os.path.join(r'R:\SCRATCH\adrianom\code\big-raster-helper\src'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
from initialize_dask import DaskHandler

handler = DaskHandler()
handler.create_local_cluster(n_workers=8, threads_per_worker=1, memory_limit="32GB")
#handler.initialize_earth_engine_dask_workers()
# If the crs is 4326 or anything geographic, I need a check to ensure that lon/lat are
# the dimension names!
# Maybe a method to close the cluster when done?

In [2]:
#http_transport = httplib2.Http(disable_ssl_certificate_validation=True)
#ee.Initialize(
#    opt_url='https://earthengine-highvolume.googleapis.com',
#    http_transport=http_transport, project='calfuels')
import json
import ee
json_key = r"R:\SCRATCH\adrianom\credentials\test-key.json"
with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)
ee.Initialize(credentials = credentials, opt_url='https://earthengine-highvolume.googleapis.com')

In [2]:
initialize_dict = {
    'opt_url': 'https://earthengine-highvolume.googleapis.com',
    'project': 'calfuels'
}

In [ ]:
# Lazy operation! Does not load the raster into memory!!
da = xarray.open_dataset("https://github.com/mapbox/rasterio/raw/1.2.1/tests/data/RGB.byte.tif")

In [ ]:
###############################
### TESTING MY PACKAGE CODE ###
###############################

In [10]:
import sys
import os
import ee
import os

ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com', project='calfuels')

def prep_sr_l8(image):
    # Develop masks for unwanted pixels (fill, cloud, cloud shadow).
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    def get_factor_img(factor_names):
        factor_list = image.toDictionary().select(factor_names).values()
        return ee.Image.constant(factor_list)
    
    scale_img = get_factor_img([
        'REFLECTANCE_MULT_BAND_.|TEMPERATURE_MULT_BAND_ST_B10'])
    offset_img = get_factor_img([
        'REFLECTANCE_ADD_BAND_.|TEMPERATURE_ADD_BAND_ST_B10'])
    scaled = image.select('SR_B.|ST_B10').multiply(scale_img).add(offset_img)

    # Replace original bands with scaled bands and apply masks.
    return image.addBands(scaled, None, True)\
        .updateMask(qa_mask).updateMask(saturation_mask)

module_path = os.path.abspath(os.path.join(r'R:\SCRATCH\adrianom\code\big-raster-helper\src'))
if module_path not in sys.path:
    sys.path.append(module_path)

from input_driver import EarthEngineData

CALIFORNIA = ee.FeatureCollection("projects/calfuels/assets/Boundaries/California")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU_remove_NV_remove_lake")

FAILING_CHUNKS = {
    'time': 48,
    'X': 512,
    'Y': 512
}
        
chunk_size = {
            'time': 3,
            'X': 2097,
            'Y': 2097
}
chunk_size  = {
            'time': 48,
            'X': 512,
            'Y': 256
}

parameters = {
            'collection': 'LANDSAT/LC08/C02/T1_L2',
            'bands': ['SR_B4', 'SR_B5'],
            'start_date': '2020-12-29',
            'end_date': '2020-12-31',
            'geometry': CALIFORNIA.geometry(),
            'crs': 'EPSG:3310',
            'scale': 100,
            'chunks': chunk_size,
            'map_function': prep_sr_l8
        }

landsat_xarray = EarthEngineData(parameters)

Task exception was never retrieved
future: <Task finished name='Task-197300' coro=<Client._gather.<locals>.wait() done, defined at r:\Users\adrianom.UNR\AppData\Local\anaconda3\envs\big-raster\lib\site-packages\distributed\client.py:2119> exception=AllExit()>
Traceback (most recent call last):
  File "r:\Users\adrianom.UNR\AppData\Local\anaconda3\envs\big-raster\lib\site-packages\distributed\client.py", line 2128, in wait
    raise AllExit()
distributed.client.AllExit
Task exception was never retrieved
future: <Task finished name='Task-197301' coro=<Client._gather.<locals>.wait() done, defined at r:\Users\adrianom.UNR\AppData\Local\anaconda3\envs\big-raster\lib\site-packages\distributed\client.py:2119> exception=AllExit()>
Traceback (most recent call last):
  File "r:\Users\adrianom.UNR\AppData\Local\anaconda3\envs\big-raster\lib\site-packages\distributed\client.py", line 2128, in wait
    raise AllExit()
distributed.client.AllExit
Task exception was never retrieved
future: <Task finis

ConnectionError: HTTPSConnectionPool(host='oauth2.googleapis.com', port=443): Max retries exceeded with url: /token (Caused by SSLError(SSLZeroReturnError(6, 'TLS/SSL connection has been closed (EOF) (_ssl.c:1133)')))

In [3]:
ds = landsat_xarray.dataset

In [30]:
chunk_size  = {
            'time': 48,
            'X': 2048,
            'Y': 1024
}
ds_rechunked = ds.chunk(chunk_size)
print(ds_rechunked)

<xarray.Dataset>
Dimensions:  (time: 3, X: 9084, Y: 10578)
Coordinates:
  * time     (time) datetime64[ns] 2020-12-29T18:57:32.281000 ... 2020-12-29T...
  * X        (X) float64 -4.216e+05 -4.215e+05 ... 4.866e+05 4.867e+05
  * Y        (Y) float64 -5.992e+05 -5.991e+05 -5.99e+05 ... 4.584e+05 4.585e+05
Data variables:
    SR_B4    (time, X, Y) float32 dask.array<chunksize=(3, 2048, 1024), meta=np.ndarray>
    SR_B5    (time, X, Y) float32 dask.array<chunksize=(3, 2048, 1024), meta=np.ndarray>
Attributes: (12/26)
    date_range:             [1365638400000, 1654560000000]
    description:            <p>This dataset contains atmospherically correcte...
    keywords:               ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'l...
    period:                 0
    product_tags:           ['global', 'sr', 'reflectance', 'l8sr', 'lst', 'c...
    provider:               USGS
    ...                     ...
    visualization_1_name:   Near Infrared (543)
    visualization_2_bands:  SR_B7,SR

In [2]:
import numpy as np
import pandas as pd
import xarray as xr

# Define the dimensions
time = pd.date_range("2020-12-29T18:57:32.281000", periods=3)
X = np.linspace(-421600, 486700, 9084)
Y = np.linspace(-599200, 458500, 10578)

# Create a data array with random data for each variable
data = np.random.rand(len(time), len(X), len(Y)).astype(np.float32)

# Create a dictionary of data variables
data_vars = {
    #'SR_B1': (['time', 'X', 'Y'], data),
    #'SR_B2': (['time', 'X', 'Y'], data),
    #'SR_B3': (['time', 'X', 'Y'], data),
    'SR_B4': (['time', 'X', 'Y'], data),
    'SR_B5': (['time', 'X', 'Y'], data),
    #'SR_B6': (['time', 'X', 'Y'], data),
    #'ST_EMSD': (['time', 'X', 'Y'], data),
    #'ST_QA': (['time', 'X', 'Y'], data),
    #'ST_TRAD': (['time', 'X', 'Y'], data),
    #'ST_URAD': (['time', 'X', 'Y'], data),
    #'QA_PIXEL': (['time', 'X', 'Y'], data),
    #'QA_RADSAT': (['time', 'X', 'Y'], data),
}

chunk_size = {'time': 3, 'X': 512, 'Y': 256}

# Create the dataset
ds = xr.Dataset(
    data_vars=data_vars,
    coords={'time': time, 'X': X, 'Y': Y},
    attrs={
        'date_range': '[1365638400000, 1654560000000]',
        'description': '<p>This dataset contains atmospherically corrected data.</p>',
        'keywords': ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'landsat'],
        'period': 0,
        'visualization_2_max': 30000.0,
        'visualization_2_min': 0.0,
        'visualization_2_name': 'Shortwave Infrared (753)',
        'crs': 'EPSG:3310'
    }
).chunk(chunk_size)


In [12]:
print(ds)

<xarray.Dataset>
Dimensions:  (time: 3, X: 9084, Y: 10578)
Coordinates:
  * time     (time) datetime64[ns] 2020-12-29T18:57:32.281000 ... 2020-12-31T...
  * X        (X) float64 -4.216e+05 -4.215e+05 ... 4.866e+05 4.867e+05
  * Y        (Y) float64 -5.992e+05 -5.991e+05 -5.99e+05 ... 4.584e+05 4.585e+05
Data variables:
    SR_B4    (time, X, Y) float32 dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
    SR_B5    (time, X, Y) float32 dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
Attributes:
    date_range:            [1365638400000, 1654560000000]
    description:           <p>This dataset contains atmospherically corrected...
    keywords:              ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'la...
    period:                0
    visualization_2_max:   30000.0
    visualization_2_min:   0.0
    visualization_2_name:  Shortwave Infrared (753)
    crs:                   EPSG:3310


In [7]:
import numpy as np
import pandas as pd
import dask.dataframe as dd

# Define the dimensions
time = pd.date_range("2020-12-29T18:57:32.281000", periods=3)
X = np.linspace(-421600, 486700, 9084)
Y = np.linspace(-599200, 458500, 10578)

# Create a data array with random data for each variable
SR_B4_data = np.random.rand(len(time), len(X), len(Y)).astype(np.float32)
SR_B5_data = np.random.rand(len(time), len(X), len(Y)).astype(np.float32)

# Create an empty list to hold DataFrame parts
df_parts = []

# Iterate through the time dimension and create DataFrame for each time step
for i, t in enumerate(time):
    df_part = pd.DataFrame({
        'time': t,
        'X': np.repeat(X, len(Y)),
        'Y': np.tile(Y, len(X)),
        'SR_B4': SR_B4_data[i].ravel(),
        'SR_B5': SR_B5_data[i].ravel()
    })
    df_parts.append(df_part)

# Concatenate all parts into a single DataFrame
df = pd.concat(df_parts)

# Convert the pandas DataFrame to a Dask DataFrame
dask_df = dd.from_pandas(df, len(time))

# Display the Dask DataFrame
#dask_df.head()


In [3]:
#################################
### Compute NDVI on an xarray ###
#################################
import dask.array as da
import xarray as xr
from dask.distributed import performance_report

def compute_ndvi(dataset):
    """
    Compute NDVI from an xarray Dataset using Dask.
    
    Parameters:
    dataset (xarray.Dataset): The input dataset containing NIR and Red bands.
    
    Returns:
    xarray.DataArray: NDVI values computed from the input dataset.
    """
    # Access the NIR (SR_B5) and Red (SR_B4) bands
    nir = dataset['SR_B5']
    red = dataset['SR_B4']
    
    # Ensure the arrays are dask arrays
    nir = nir.data if isinstance(nir, xr.DataArray) else nir
    red = red.data if isinstance(red, xr.DataArray) else red
    
    # Compute NDVI using Dask array operations
    ndvi = (nir - red) / (nir + red)
    
    # Convert the resulting dask array back to a DataArray
    ndvi_da = xr.DataArray(ndvi, dims=dataset['SR_B5'].dims, coords=dataset['SR_B5'].coords, name='NDVI')
    dataset['NDVI'] = ndvi_da
    
    return dataset

# Example usage with the dataset created earlier
ndvi = compute_ndvi(dataset)
ndvi.compute()
#with performance_report(filename='dask-report-nodf.html'):
#    ndvi.compute()

<xarray.Dataset>
Dimensions:  (time: 3, X: 9084, Y: 10578)
Coordinates:
  * time     (time) datetime64[ns] 2020-12-29T18:57:32.281000 ... 2020-12-31T...
  * X        (X) float64 -4.216e+05 -4.215e+05 ... 4.866e+05 4.867e+05
  * Y        (Y) float64 -5.992e+05 -5.991e+05 -5.99e+05 ... 4.584e+05 4.585e+05
Data variables:
    SR_B4    (time, X, Y) float32 0.08238 0.2047 0.7059 ... 0.8569 0.1792 0.9919
    SR_B5    (time, X, Y) float32 0.08238 0.2047 0.7059 ... 0.8569 0.1792 0.9919
    NDVI     (time, X, Y) float32 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
Attributes:
    date_range:            [1365638400000, 1654560000000]
    description:           <p>This dataset contains atmospherically corrected...
    keywords:              ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'la...
    period:                0
    visualization_2_max:   30000.0
    visualization_2_min:   0.0
    visualization_2_name:  Shortwave Infrared (753)
    crs:                   EPSG:3310

In [9]:
#################################
### Compute NDVI on an xarray ###
#################################
import dask.array as da
import xarray as xr
from dask.distributed import performance_report

def compute_ndvi(dataset):
    """
    Compute NDVI from an xarray Dataset using Dask.
    
    Parameters:
    dataset (xarray.Dataset): The input dataset containing NIR and Red bands.
    
    Returns:
    xarray.DataArray: NDVI values computed from the input dataset.
    """
    #ds = dataset.load()
    #print(dataset)
    df = dataset.to_dataframe().reset_index()
    print(dataset)
    print(df)

    #df['ndvi'] = (df['nir'] - df['red']) / (df['nir'] + df['red'])
    
    return df

# Example usage with the dataset created earlier
ndvi = compute_ndvi(dataset)
#ndvi.compute()
#with performance_report(filename='dask-report-nodf.html'):
#    ndvi.compute()

<xarray.Dataset>
Dimensions:  (time: 3, X: 9084, Y: 10578)
Coordinates:
  * time     (time) datetime64[ns] 2020-12-29T18:57:32.281000 ... 2020-12-31T...
  * X        (X) float64 -4.216e+05 -4.215e+05 ... 4.866e+05 4.867e+05
  * Y        (Y) float64 -5.992e+05 -5.991e+05 -5.99e+05 ... 4.584e+05 4.585e+05
Data variables:
    SR_B4    (time, X, Y) float32 dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
    SR_B5    (time, X, Y) float32 dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
Attributes:
    date_range:            [1365638400000, 1654560000000]
    description:           <p>This dataset contains atmospherically corrected...
    keywords:              ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'la...
    period:                0
    visualization_2_max:   30000.0
    visualization_2_min:   0.0
    visualization_2_name:  Shortwave Infrared (753)
    crs:                   EPSG:3310
                             time         X         Y     SR_B4     SR_B5
0         2020-

In [3]:
import xarray as xr
import numpy as np
import dask.array as da

chunk_size = ds['SR_B4'].chunks

# Create a template with dask arrays using the same chunk sizes and attributes
template = xr.Dataset(
    {
        'SR_B4': (['time', 'X', 'Y'], da.empty((3, 9084, 10578), chunks=chunk_size, dtype=np.float32)),
        'SR_B5': (['time', 'X', 'Y'], da.empty((3, 9084, 10578), chunks=chunk_size, dtype=np.float32)),
        #'ndvi':  (['time', 'X', 'Y'], da.empty((3, 9084, 10578), chunks=chunk_size, dtype=np.float32))
    },
    coords={
        'time': ds.coords['time'],
        'X': ds.coords['X'],
        'Y': ds.coords['Y'],
    },
    attrs=ds.attrs  # Copy over the original dataset's attributes
)

In [3]:
import xarray as xr
import pandas as pd

def process_chunk_as_pandas(chunk):
    # Convert xarray chunk to pandas DataFrame
    df = chunk.to_dataframe().reset_index()
    
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    # Convert the pandas DataFrame back to an xarray structure
    df = df.set_index(list(chunk.dims))
    result = df.to_xarray()
    
    return result

In [26]:
# Create a function to generate the template based on the original dataset
def create_template(ds, process_func):
    # Extract a single chunk to determine the output structure
    one_chunk = ds.isel(time=slice(0, ds.chunks['time'][0]), 
                        X=slice(0, ds.chunks['X'][0]), 
                        Y=slice(0, ds.chunks['Y'][0]))
    
    # Apply the processing function to this chunk
    processed_chunk = process_func(one_chunk)
    
    # Create the template using a combination of original data variables and newly created ones
    template_vars = {}
    
    for var in processed_chunk.data_vars:
        if var in ds.data_vars:
            # Use the original dataset's shape and chunking for existing variables
            template_vars[var] = (processed_chunk[var].dims, 
                                  da.empty(ds[var].shape, 
                                           chunks=ds[var].chunks, 
                                           dtype=processed_chunk[var].dtype))
        else:
            # For new variables, define the shape and chunks manually based on the original chunking strategy
            new_var_shape = tuple(ds.dims[dim] for dim in processed_chunk[var].dims)
            new_var_chunks = tuple(ds.chunks[dim][0] for dim in processed_chunk[var].dims)
            template_vars[var] = (processed_chunk[var].dims, 
                                  da.empty(new_var_shape, 
                                           chunks=new_var_chunks, 
                                           dtype=processed_chunk[var].dtype))
    
    template = xr.Dataset(
        template_vars,
        coords={coord: ds.coords[coord] for coord in ds.coords},
        attrs=ds.attrs
    )
    
    return template

# Assuming `ds` is your chunked xarray object:
template = create_template(ds, process_chunk_as_pandas)

In [28]:
# Assuming `your_chunked_xarray` is your chunked xarray object:
result = xr.map_blocks(process_chunk_as_pandas, ds, template=template)
result.compute()

<xarray.Dataset>
Dimensions:  (time: 3, X: 9084, Y: 10578)
Coordinates:
  * time     (time) datetime64[ns] 2020-12-29T18:57:32.281000 ... 2020-12-31T...
  * X        (X) float64 -4.216e+05 -4.215e+05 ... 4.866e+05 4.867e+05
  * Y        (Y) float64 -5.992e+05 -5.991e+05 -5.99e+05 ... 4.584e+05 4.585e+05
Data variables:
    SR_B4    (time, X, Y) float32 0.411 0.9277 0.7535 ... 0.5931 0.7941 0.09407
    SR_B5    (time, X, Y) float32 0.411 0.9277 0.7535 ... 0.5931 0.7941 0.09407
    ndvi     (time, X, Y) float32 0.822 1.855 1.507 1.859 ... 1.186 1.588 0.1881
Attributes:
    date_range:            [1365638400000, 1654560000000]
    description:           <p>This dataset contains atmospherically corrected...
    keywords:              ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'la...
    period:                0
    visualization_2_max:   30000.0
    visualization_2_min:   0.0
    visualization_2_name:  Shortwave Infrared (753)
    crs:                   EPSG:3310

In [4]:
##############################
### A SINGLE CHUNK OF DATA ###
##############################
dataset = landsat_xarray.dataset

# Select the first chunk (first time step, top-left corner) for SR_B5 and SR_B4
nir_chunk = dataset['SR_B5'].data.blocks[0, 0, 0]
red_chunk = dataset['SR_B4'].data.blocks[0, 0, 0]

# Define the NDVI computation function
def compute_ndvi(nir, red):
    return (nir - red) / (nir + red)

# Compute the NDVI for the selected chunk
ndvi_chunk = compute_ndvi(nir_chunk, red_chunk)

ndvi_chunk.compute()
# Visualize the task graph
#ndvi_chunk.visualize(filename='ndvi_chunk_task_graph_small.png')

In [3]:
########################################################################
### Convert to data frame, run function, then convert back to xarray ###
########################################################################
def compute_ndvi(dataset):
    dask_df = dataset.to_dask_dataframe()
    # Ensure that the required columns are in the DataFrame
    if 'SR_B5' not in dask_df.columns or 'SR_B4' not in dask_df.columns:
        raise ValueError("DataFrame must contain 'SR_B5' and 'SR_B4' columns")

    # Compute NDVI
    dask_df['NDVI'] = (dask_df['SR_B5'] - dask_df['SR_B4']) / (dask_df['SR_B5'] + dask_df['SR_B4'])
    return dask_df
ndvi = compute_ndvi(dataset)

r:\Users\adrianom.UNR\AppData\Local\anaconda3\envs\big-raster\lib\site-packages\IPython\core\interactiveshell.py:3550: PerformanceWarning: Reshaping is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array.reshape(shape)

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array.reshape(shape)Explictly passing ``limit`` to ``reshape`` will also silence this warning
    >>> array.reshape(shape, limit='128 MiB')
  exec(code_obj, self.user_global_ns, self.user_ns)
r:\Users\adrianom.UNR\AppData\Local\anaconda3\envs\big-raster\lib\site-packages\IPython\core\interactiveshell.py:3550: PerformanceWarning: Reshaping is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': Fa

In [5]:
########################################
### Compute NDVI on a Dask Dataframe ###
########################################

#dask_df = dataset.to_dask_dataframe().repartition(100)

def compute_ndvi(dask_df):
    # Ensure that the required columns are in the DataFrame
    if 'SR_B5' not in dask_df.columns or 'SR_B4' not in dask_df.columns:
        raise ValueError("DataFrame must contain 'SR_B5' and 'SR_B4' columns")

    # Compute NDVI
    dask_df['NDVI'] = (dask_df['SR_B5'] - dask_df['SR_B4']) / (dask_df['SR_B5'] + dask_df['SR_B4'])

    return dask_df

result = compute_ndvi(dask_df)
result.compute()

ValueError: Shape of passed values is (2, 1), indices imply (33554432, 1)

In [ ]:
dask_df = handler.dataset_to_dask_dataframe(landsat_xarray.dataset)

def compute_ndvi(dask_df):
    # Ensure that the required columns are in the DataFrame
    if 'SR_B5' not in dask_df.columns or 'SR_B4' not in dask_df.columns:
        raise ValueError("DataFrame must contain 'SR_B5' and 'SR_B4' columns")

    # Compute NDVI
    dask_df['NDVI'] = (dask_df['SR_B5'] - dask_df['SR_B4']) / (dask_df['SR_B5'] + dask_df['SR_B4'])

    return dask_df

from udf_tuner import UserDefinedFunctionTuner
tuner = UserDefinedFunctionTuner(dask_df)
tuner.apply_function(compute_ndvi)

In [3]:
import xarray as xr
import pandas as pd

def process_chunk_as_pandas(df):
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    return df

In [4]:
import sys

module_path = os.path.abspath(os.path.join(r'R:\SCRATCH\adrianom\code\big-raster-helper\src'))
if module_path not in sys.path:
    sys.path.append(module_path)

from udf_tuner import UserDefinedFunction

user_defined_func = UserDefinedFunction()

# Apply the user-defined function
result = user_defined_func.apply_user_function(ds, process_chunk_as_pandas)

In [5]:
result.compute()

<xarray.Dataset>
Dimensions:  (time: 3, X: 9084, Y: 10578)
Coordinates:
  * time     (time) datetime64[ns] 2020-12-29T18:57:32.281000 ... 2020-12-31T...
  * X        (X) float64 -4.216e+05 -4.215e+05 ... 4.866e+05 4.867e+05
  * Y        (Y) float64 -5.992e+05 -5.991e+05 -5.99e+05 ... 4.584e+05 4.585e+05
Data variables:
    SR_B4    (time, X, Y) float32 0.03795 0.8431 0.02565 ... 0.6655 0.09735
    SR_B5    (time, X, Y) float32 0.03795 0.8431 0.02565 ... 0.6655 0.09735
    ndvi     (time, X, Y) float32 0.07589 1.686 0.0513 ... 1.807 1.331 0.1947
Attributes:
    date_range:            [1365638400000, 1654560000000]
    description:           <p>This dataset contains atmospherically corrected...
    keywords:              ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'la...
    period:                0
    visualization_2_max:   30000.0
    visualization_2_min:   0.0
    visualization_2_name:  Shortwave Infrared (753)
    crs:                   EPSG:3310

In [5]:
def compute_ndvi(dask_df):
    # Ensure that the required columns are in the DataFrame
    if 'SR_B5' not in dask_df.columns or 'SR_B4' not in dask_df.columns:
        raise ValueError("DataFrame must contain 'SR_B5' and 'SR_B4' columns")

    # Compute NDVI
    dask_df['NDVI'] = (dask_df['SR_B5'] - dask_df['SR_B4']) / (dask_df['SR_B5'] + dask_df['SR_B4'])

    return dask_df

def apply_function(dataset, func, *args, **kwargs):
    """
    Apply a user-defined function to the Dask DataFrame.
        
    Parameters:
    - func: the user-defined function to apply.
    - args: positional arguments to pass to the function.
    - kwargs: keyword arguments to pass to the function.
        
    Returns:
    - result: the result of applying the function to the dataframe.
    """
    if not callable(func):
        raise ValueError("The provided function must be callable.")
    
    dask_df = dataset.to_dask_dataframe()
    result = dask_df.map_partitions(func, *args, **kwargs)
    result.compute()
    return result

    #report_name="report"
    #with performance_report(filename=f"{report_name}.html"):
    #    result.compute()
result = apply_function(dataset, compute_ndvi)

r:\Users\adrianom.UNR\AppData\Local\anaconda3\envs\big-raster\lib\site-packages\IPython\core\interactiveshell.py:3550: PerformanceWarning: Reshaping is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array.reshape(shape)

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array.reshape(shape)Explictly passing ``limit`` to ``reshape`` will also silence this warning
    >>> array.reshape(shape, limit='128 MiB')
  exec(code_obj, self.user_global_ns, self.user_ns)
r:\Users\adrianom.UNR\AppData\Local\anaconda3\envs\big-raster\lib\site-packages\IPython\core\interactiveshell.py:3550: PerformanceWarning: Reshaping is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': Fa